# U.S. Analytics Job Market — Skill Extraction & Entry-Level Validation

## Objective

This notebook enriches the cleaned job dataset with:

- Technical skill signals extracted from job titles and descriptions
- Entry-level / new-grad signals derived from explicit keywords, experience requirements, and title seniority

The resulting fields are treated as transparent **heuristics**, not perfect ground-truth labels.

## 1. Environment Setup

In [26]:
from pathlib import Path
import re
import pandas as pd

## 2. Load the Latest Processed Job Snapshot

Parquet is preferred because it preserves data types. CSV is used as a fallback.

In [27]:
candidate_dirs = [Path("../data/processed"), Path("data/processed")]
processed_dir = next((p for p in candidate_dirs if p.exists()), candidate_dirs[0])

parquet_files = sorted(
    p
    for p in processed_dir.glob("us_analytics_jobs_*.parquet")
    if "enriched" not in p.name
)
csv_files = sorted(
    p for p in processed_dir.glob("us_analytics_jobs_*.csv") if "enriched" not in p.name
)

if parquet_files:
    source_path = parquet_files[-1]
    jobs_df = pd.read_parquet(source_path)
elif csv_files:
    source_path = csv_files[-1]
    jobs_df = pd.read_csv(source_path)
else:
    raise FileNotFoundError("No processed job snapshot found. Run Notebook 2 first.")

print("Loaded:", source_path)
print("Shape:", jobs_df.shape)

Loaded: ../data/processed/us_analytics_jobs_2026-09-29.parquet
Shape: (750, 21)


### 2.1 Validate Required Columns

In [28]:
required_columns = [
    "id",
    "created",
    "search_role",
    "title",
    "company",
    "description",
    "state",
    "location",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_is_predicted",
    "salary_source",
]

missing_required = [c for c in required_columns if c not in jobs_df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("Required columns validated.")
print("Rows:", len(jobs_df))
print("Unique job IDs:", jobs_df["id"].nunique())
print("Duplicate job IDs:", jobs_df["id"].duplicated().sum())

Required columns validated.
Rows: 750
Unique job IDs: 750
Duplicate job IDs: 0


## 3. Normalize Text for Pattern Matching

Create a capitalization-preserving text field and a lowercased normalized field for regex matching.

In [29]:
def normalize_text(series):
    return (
        series.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    )


jobs_df["title"] = normalize_text(jobs_df["title"])
jobs_df["description"] = normalize_text(jobs_df["description"])

jobs_df["search_text_raw"] = (
    jobs_df["title"] + " " + jobs_df["description"]
).str.strip()

jobs_df["search_text"] = jobs_df["search_text_raw"].str.lower()

## 4. Define the Skill Dictionary

Word boundaries and explicit product names reduce false positives.  
The programming language **R** is matched separately because naive case-insensitive single-letter matching is unsafe.

In [30]:
skill_patterns = {
    "sql": r"\b(?:sql|mysql|postgresql|t-sql)\b",
    "python": r"\bpython\b",
    "excel": r"\b(?:microsoft\s+)?excel\b",
    "tableau": r"\btableau\b",
    "power_bi": r"\bpower\s*bi\b",
    "looker": r"\blooker(?:\s+studio)?\b",
    "aws": r"\b(?:aws|amazon\s+web\s+services)\b",
    "gcp": r"\b(?:gcp|google\s+cloud(?:\s+platform)?)\b",
    "azure": r"\b(?:azure|microsoft\s+azure)\b",
    "snowflake": r"\bsnowflake\b",
    "spark": r"\b(?:apache\s+spark|spark)\b",
    "pyspark": r"\bpyspark\b",
    "databricks": r"\bdatabricks\b",
    "git": r"\bgit(?:hub|lab)?\b",
}

r_pattern = r"(?<![A-Za-z0-9_])R(?![A-Za-z0-9_])|(?i:\brstudio\b)"

## 5. Extract Skill Flags

In [31]:
skill_columns = []

for skill, pattern in skill_patterns.items():
    col = f"skill_{skill}"
    jobs_df[col] = (
        jobs_df["search_text"]
        .str.contains(pattern, regex=True, na=False)
        .astype("int8")
    )
    skill_columns.append(col)

jobs_df["skill_r"] = (
    jobs_df["search_text_raw"]
    .str.contains(r_pattern, regex=True, na=False)
    .astype("int8")
)
skill_columns.append("skill_r")

print("Skill flags created:", len(skill_columns))

Skill flags created: 15


### 5.1 Overall Skill Prevalence

In [32]:
skill_summary = pd.DataFrame(
    {
        "skill": [c.replace("skill_", "") for c in skill_columns],
        "job_count": [int(jobs_df[c].sum()) for c in skill_columns],
        "share_pct": [round(jobs_df[c].mean() * 100, 1) for c in skill_columns],
    }
).sort_values(["job_count", "skill"], ascending=[False, True], ignore_index=True)

skill_summary

,skill,job_count,share_pct
0,sql,58,7.7
1,power_bi,31,4.1
2,tableau,14,1.9
3,excel,13,1.7
4,python,13,1.7
5,azure,7,0.9
6,aws,4,0.5
7,r,3,0.4
8,snowflake,3,0.4
9,databricks,2,0.3


### 5.2 Skill Prevalence by Search Role

In [33]:
skill_by_role = jobs_df.groupby("search_role")[skill_columns].mean().mul(100).round(1).T
skill_by_role.index = skill_by_role.index.str.replace("skill_", "", regex=False)
skill_by_role

search_role,business analyst,data analyst,product analyst
sql,5.2,14.0,4.0
python,0.4,4.8,0.0
excel,1.6,3.6,0.0
tableau,1.2,4.4,0.0
power_bi,2.4,10.0,0.0
looker,0.0,0.0,0.0
aws,0.4,1.2,0.0
gcp,0.0,0.0,0.0
azure,0.8,1.6,0.4
snowflake,0.0,1.2,0.0


## 6. Validate Skill Matches

Inspect deterministic matched and unmatched samples before relying on keyword flags.

In [34]:
def sample_skill_records(df, flag_column, matched=True, n=5):
    value = 1 if matched else 0
    sample = df.loc[
        df[flag_column].eq(value),
        ["id", "search_role", "title", "company", "description"],
    ].copy()
    sample["description_excerpt"] = sample["description"].str.slice(0, 220)
    return sample.drop(columns="description").head(n).reset_index(drop=True)

### 6.1 SQL Match Sample

In [35]:
sample_skill_records(jobs_df, "skill_sql", matched=True, n=5)

,id,search_role,title,company,description_excerpt
0,5898910111,data analyst,Data Analyst,Kforce Technology Staffing,RESPONSIBILITIES: Kforce's retail client in No...
1,5897303369,data analyst,Data Analyst,Turing IT Labs,"Data Analyst Manhattan, NYC Location: Manhatta..."
2,5775687128,data analyst,Data Analyst,"Judge Group, Inc.","Location: Bridgeport, PA Salary: $70,000.00 US..."
3,5769854278,data analyst,Data Analyst,Enterprise Business Solutions Inc,Job Summary We are seeking a Data Analyst to c...
4,5886626110,data analyst,Data Analyst,"Mitchell Martin, Inc.","Data Analyst - Charlotte, NC - Contract - $47-..."


### 6.2 Power BI Match Sample

In [36]:
sample_skill_records(jobs_df, "skill_power_bi", matched=True, n=5)

,id,search_role,title,company,description_excerpt
0,5901431344,data analyst,Data Analyst/Power Platform,Guidehouse,Job Family : Data Science Consulting Travel Re...
1,5897303369,data analyst,Data Analyst,Turing IT Labs,"Data Analyst Manhattan, NYC Location: Manhatta..."
2,5775687128,data analyst,Data Analyst,"Judge Group, Inc.","Location: Bridgeport, PA Salary: $70,000.00 US..."
3,5769854278,data analyst,Data Analyst,Enterprise Business Solutions Inc,Job Summary We are seeking a Data Analyst to c...
4,5880286528,data analyst,Data Analyst,Apex Systems,Job: 3051820 Job Description: Data Analyst Loc...


### 6.3 R Match Sample

R receives extra QA because single-letter matching is especially prone to false positives.

In [37]:
sample_skill_records(jobs_df, "skill_r", matched=True, n=10)

,id,search_role,title,company,description_excerpt
0,5898790665,data analyst,Data Analyst,QUANTUM TECHNOLOGIES LLC,"Job Title:Data Analyst Location: McLean, VA Du..."
1,5850130361,data analyst,Data Analyst/Engineer,"Stanfield Systems, Inc.",Qualifications - Bachelor''s degree in Health ...
2,5886848652,business analyst,Functional Business Analyst,"Techno-Comp, Inc.","Hi, This is Srikanth from Techno-Comp Inc. We ..."


### 6.4 SQL Non-Match Sample

In [38]:
sample_skill_records(jobs_df, "skill_sql", matched=False, n=5)

,id,search_role,title,company,description_excerpt
0,5901208184,data analyst,Data Analyst,Guidehouse,Job Family : Data Science Consulting Travel Re...
1,5896874781,data analyst,Data analyst,Vaco LLC,"Data Analyst Location: Hybrid | Nashville, TN ..."
2,5886631734,data analyst,Human Resources Data Analyst,Arnold Porter,Arnold & Porter is an international law firm w...
3,5886391534,data analyst,Data Analyst (Financials data Analyst),"eSolutionsFirst, LLC","Hi, We currently have an exciting opportunity ..."
4,5888579132,data analyst,Data Analyst,MANTECH,hackajob is collaborating with MANTECH to conn...


## 7. Entry-Level / New-Grad Classification

The classification uses multiple auditable signals rather than one opaque label:

- Explicit new-grad language
- Explicit entry-level / early-career language
- Low experience requirements
- Higher experience requirements
- Senior title language
- Internship language

In [39]:
new_grad_pattern = (
    r"\b(?:new\s+grad(?:uate)?s?|recent\s+grad(?:uate)?s?|"
    r"graduate\s+(?:program|programme|analyst|role|position)|campus\s+hire)\b"
)

entry_keyword_pattern = (
    r"\b(?:entry[-\s]?level|early[-\s]?career|"
    r"junior\s+(?:data|business|product|analytics?)\s+analyst)\b"
)

low_experience_pattern = (
    r"\b(?:0|1|2)\+?\s+years?(?:\s+of)?(?:\s+\w+){0,4}\s+experience\b|"
    r"\b(?:0|1)\s*(?:-|–|—|to)\s*2\s+years?"
    r"(?:\s+of)?(?:\s+\w+){0,4}\s+experience\b|"
    r"\b(?:up\s+to|maximum(?:\s+of)?|max(?:imum)?\.?)\s*2\s+years?"
    r"(?:\s+of)?(?:\s+\w+){0,4}\s+experience\b"
)

high_experience_pattern = (
    r"\b(?:3|4|5|6|7|8|9|10|1[1-9]|2[0-9])\+?\s+years?"
    r"(?:\s+of)?(?:\s+\w+){0,4}\s+experience\b"
)

senior_title_pattern = (
    r"\b(?:senior|sr\.?|lead|principal|staff|manager|director|head)\b"
)
internship_pattern = r"\b(?:intern|internship|co[-\s]?op)\b"

### 7.1 Create Component Career-Stage Flags

In [40]:
jobs_df["new_grad_keyword_flag"] = (
    jobs_df["search_text"]
    .str.contains(new_grad_pattern, regex=True, na=False)
    .astype("int8")
)

jobs_df["entry_keyword_flag"] = (
    jobs_df["search_text"]
    .str.contains(entry_keyword_pattern, regex=True, na=False)
    .astype("int8")
)

jobs_df["low_experience_flag"] = (
    jobs_df["search_text"]
    .str.contains(low_experience_pattern, regex=True, na=False)
    .astype("int8")
)

jobs_df["high_experience_flag"] = (
    jobs_df["search_text"]
    .str.contains(high_experience_pattern, regex=True, na=False)
    .astype("int8")
)

jobs_df["senior_title_flag"] = (
    jobs_df["title"]
    .str.lower()
    .str.contains(senior_title_pattern, regex=True, na=False)
    .astype("int8")
)

jobs_df["internship_flag"] = (
    jobs_df["title"]
    .str.lower()
    .str.contains(internship_pattern, regex=True, na=False)
    .astype("int8")
)

### 7.2 Extract Minimum Experience Years Mentioned

This helper extracts the lowest explicit experience-years number found in relevant phrases.  
It is a text-derived signal, not a guaranteed overall requirement.

In [41]:
experience_range_regex = re.compile(
    r"\b(\d{1,2})\s*(?:-|–|—|to)\s*(\d{1,2})\s+years?"
    r"(?:\s+of)?(?:\s+\w+){0,4}\s+experience\b",
    flags=re.IGNORECASE,
)

experience_single_regex = re.compile(
    r"\b(\d{1,2})\+?\s+years?"
    r"(?:\s+of)?(?:\s+\w+){0,4}\s+experience\b",
    flags=re.IGNORECASE,
)


def extract_min_experience_years(text):
    if not isinstance(text, str) or not text:
        return pd.NA

    years = []
    years += [int(m.group(1)) for m in experience_range_regex.finditer(text)]
    years += [int(m.group(1)) for m in experience_single_regex.finditer(text)]

    return min(years) if years else pd.NA


jobs_df["min_experience_years_mentioned"] = (
    jobs_df["search_text_raw"].apply(extract_min_experience_years).astype("Int64")
)

### 7.3 Create a Conservative Entry-Level Flag

A posting is flagged as entry-level only when an entry/new-grad or low-experience signal is present **without** a conflicting senior-title, 3+ year experience, or internship signal.

Potential contradictions are retained separately in `entry_conflict_flag` for QA rather than being counted as confirmed entry-level records.

In [42]:
explicit_entry_signal = jobs_df["new_grad_keyword_flag"].eq(1) | jobs_df[
    "entry_keyword_flag"
].eq(1)

low_experience_entry_signal = jobs_df["low_experience_flag"].eq(1)

conflicting_experience_signal = jobs_df["high_experience_flag"].eq(1) | jobs_df[
    "senior_title_flag"
].eq(1)

# Explicit entry-level language that conflicts with
# senior-title or 3+ years experience signals
jobs_df["entry_conflict_flag"] = (
    explicit_entry_signal & conflicting_experience_signal
).astype("int8")

# Conservative entry-level classification
jobs_df["entry_level_flag"] = (
    (explicit_entry_signal | low_experience_entry_signal)
    & ~conflicting_experience_signal
    & jobs_df["internship_flag"].eq(0)
).astype("int8")


In [43]:
def classify_entry_basis(row):
    if row["internship_flag"] == 1:
        return "internship"
    if row["entry_conflict_flag"] == 1:
        return "conflicting_entry_signal"
    if row["new_grad_keyword_flag"] == 1:
        return "new_grad_keyword"
    if row["entry_keyword_flag"] == 1:
        return "entry_keyword"
    if (
        row["low_experience_flag"] == 1
        and row["high_experience_flag"] == 0
        and row["senior_title_flag"] == 0
    ):
        return "low_experience_requirement"
    if row["senior_title_flag"] == 1:
        return "senior_title_signal"
    if row["high_experience_flag"] == 1:
        return "3plus_experience_signal"
    return "no_clear_signal"


jobs_df["entry_classification_basis"] = jobs_df.apply(
    classify_entry_basis,
    axis=1,
)

jobs_df["career_stage_signal"] = "unspecified"

jobs_df.loc[
    jobs_df["high_experience_flag"].eq(1),
    "career_stage_signal",
] = "experienced_signal"

jobs_df.loc[
    jobs_df["entry_level_flag"].eq(1),
    "career_stage_signal",
] = "entry_level_signal"

jobs_df.loc[
    jobs_df["entry_conflict_flag"].eq(1),
    "career_stage_signal",
] = "ambiguous_entry_signal"

jobs_df.loc[
    jobs_df["senior_title_flag"].eq(1),
    "career_stage_signal",
] = "senior_title_signal"

jobs_df.loc[
    jobs_df["internship_flag"].eq(1),
    "career_stage_signal",
] = "internship"

## 8. Validate Entry-Level Classification

In [44]:
entry_summary = pd.Series(
    {
        "entry_level_flagged": int(jobs_df["entry_level_flag"].sum()),
        "entry_conflicts": int(jobs_df["entry_conflict_flag"].sum()),
        "new_grad_keyword": int(jobs_df["new_grad_keyword_flag"].sum()),
        "entry_keyword": int(jobs_df["entry_keyword_flag"].sum()),
        "low_experience_signal": int(jobs_df["low_experience_flag"].sum()),
        "high_experience_signal": int(jobs_df["high_experience_flag"].sum()),
        "senior_title_signal": int(jobs_df["senior_title_flag"].sum()),
        "internship_signal": int(jobs_df["internship_flag"].sum()),
    }
)

entry_summary

entry_level_flagged         9
entry_conflicts            10
new_grad_keyword            0
entry_keyword              16
low_experience_signal       4
high_experience_signal     54
senior_title_signal       108
internship_signal           4
dtype: int64

In [45]:
entry_by_role = (
    jobs_df.groupby("search_role")["entry_level_flag"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "entry_level_jobs"})
)
entry_by_role["entry_level_share_pct"] = (
    entry_by_role["entry_level_jobs"] / entry_by_role["count"] * 100
).round(1)
entry_by_role

,entry_level_jobs,count,entry_level_share_pct
search_role,,,
business analyst,6,250,2.4
data analyst,3,250,1.2
product analyst,0,250,0.0


### 8.1 Review Flagged Entry-Level Examples

In [46]:
entry_review = jobs_df.loc[
    jobs_df["entry_level_flag"].eq(1),
    [
        "id",
        "search_role",
        "title",
        "company",
        "min_experience_years_mentioned",
        "entry_classification_basis",
        "description",
    ],
].copy()

entry_review["description_excerpt"] = entry_review["description"].str.slice(0, 260)
entry_review.drop(columns="description").head(15)

,id,search_role,title,company,min_experience_years_mentioned,entry_classification_basis,description_excerpt
172,5850130361,data analyst,Data Analyst/Engineer,"Stanfield Systems, Inc.",2,low_experience_requirement,Qualifications - Bachelor''s degree in Health ...
226,5894385053,data analyst,Entry-Level Data Analyst,Dhalite Inc.,<NA>,entry_keyword,We are looking for a motivated and detail-orie...
231,5901757553,data analyst,Entry level Data Analyst,Dhalite Inc.,<NA>,entry_keyword,Note: Applicants must be currently authorized ...
266,5867170928,business analyst,Business Analyst,StarsHR Inc,2,low_experience_requirement,Minimum 2 years of experience as a Business An...
313,5893064699,business analyst,Business Analyst,Siritek Inc,2,low_experience_requirement,Siritek Inc. is seeking a skilled and detail-o...
402,5894500998,business analyst,Junior Business Analyst,Dhalite Inc.,0,entry_keyword,"Junior Business Analyst Location: Boston, MA J..."
417,5875731316,business analyst,Junior Business Analyst,Ellianse LLC,<NA>,entry_keyword,Position: Junior Business Analyst Full Time Mu...
456,5893126745,business analyst,junior business analyst,Dhalite Inc.,<NA>,entry_keyword,Summary: We are seeking a Business Analyst I t...
494,5901903922,business analyst,Junior Business Analyst,Dhalite Inc.,<NA>,entry_keyword,Summary: We are looking for a detail-oriented ...


### 8.2 Review Classification Conflicts

Explicit entry-level language can occasionally coexist with senior-title or 3+ year experience signals. These records are marked as ambiguous and excluded from the conservative `entry_level_flag`.

In [47]:
entry_conflicts = jobs_df.loc[
    jobs_df["entry_conflict_flag"].eq(1),
    [
        "id",
        "search_role",
        "title",
        "company",
        "entry_classification_basis",
        "min_experience_years_mentioned",
        "description",
    ],
].copy()

entry_conflicts["description_excerpt"] = entry_conflicts["description"].str.slice(
    0, 260
)

entry_conflicts.drop(columns="description").head(15)

,id,search_role,title,company,entry_classification_basis,min_experience_years_mentioned,description_excerpt
22,5889610205,data analyst,DATA ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Data Analyst Retail & E-Commerce ( 2-3 Years E...
23,5894163152,data analyst,DATA ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Data Analyst Retail & E-Commerce (2-3 Years Ex...
26,5895253703,data analyst,DATA ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Data Analyst Healthcare (2-3 Years Experience)...
27,5892848686,data analyst,DATA ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Data Analyst Healthcare (2-3 Years Experience)...
28,5901736407,data analyst,DATA ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Data Analyst Healthcare (2-3 Years Experience)...
263,5894159259,business analyst,BUSINESS ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Business Analyst Retail & E-Commerce (2-3 Year...
265,5889617759,business analyst,BUSINESS ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Business Analyst Retail & E-Commerce (2-3 Year...
267,5901736211,business analyst,BUSINESS ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Business Analyst Healthcare ( 2-3 Years Experi...
268,5895261180,business analyst,BUSINESS ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Business Analyst Healthcare (2-3 Years Experie...
269,5892848687,business analyst,BUSINESS ANALYST,AaraTechnologies Inc,conflicting_entry_signal,2,Business Analyst Healthcare (2-3 Years Experie...


## 9. Create a Long-Form Skill Table

The wide job-level table is useful for dashboards, while the long-form table is convenient for SQL skill analysis.

In [48]:
skill_long_df = jobs_df[["id", "search_role", "title", "company"] + skill_columns].melt(
    id_vars=["id", "search_role", "title", "company"],
    value_vars=skill_columns,
    var_name="skill",
    value_name="skill_present",
)

skill_long_df = skill_long_df.loc[skill_long_df["skill_present"].eq(1)].copy()

skill_long_df["skill"] = skill_long_df["skill"].str.replace("skill_", "", regex=False)

skill_long_df = skill_long_df.reset_index(drop=True)
skill_long_df.head()

,id,search_role,title,company,skill,skill_present
0,5898910111,data analyst,Data Analyst,Kforce Technology Staffing,sql,1
1,5897303369,data analyst,Data Analyst,Turing IT Labs,sql,1
2,5775687128,data analyst,Data Analyst,"Judge Group, Inc.",sql,1
3,5769854278,data analyst,Data Analyst,Enterprise Business Solutions Inc,sql,1
4,5886626110,data analyst,Data Analyst,"Mitchell Martin, Inc.",sql,1


## 10. Build the Final Enriched Job Dataset

In [49]:
base_columns = [
    "id",
    "created",
    "ingested_at_utc",
    "search_role",
    "matched_search_roles",
    "title",
    "company",
    "description",
    "country",
    "state",
    "location",
    "location_hierarchy",
    "location_depth",
    "latitude",
    "longitude",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_is_predicted",
    "salary_source",
    "data_source",
]

career_columns = [
    "new_grad_keyword_flag",
    "entry_keyword_flag",
    "low_experience_flag",
    "high_experience_flag",
    "senior_title_flag",
    "internship_flag",
    "min_experience_years_mentioned",
    "entry_level_flag",
    "entry_conflict_flag",
    "entry_classification_basis",
    "career_stage_signal",
]

final_columns = (
    [c for c in base_columns if c in jobs_df.columns] + skill_columns + career_columns
)

enriched_df = jobs_df[final_columns].copy()
enriched_df.shape

(750, 47)

## 11. Final Quality Validation

In [50]:
final_quality = pd.Series(
    {
        "rows": len(enriched_df),
        "unique_job_ids": enriched_df["id"].nunique(),
        "duplicate_job_ids": enriched_df["id"].duplicated().sum(),
        "missing_job_ids": enriched_df["id"].isna().sum(),
        "missing_titles": enriched_df["title"].isna().sum(),
        "missing_descriptions": enriched_df["description"].isna().sum(),
        "entry_level_jobs": int(enriched_df["entry_level_flag"].sum()),
        "skill_mentions_total": int(enriched_df[skill_columns].sum().sum()),
    }
)
final_quality

rows                    750
unique_job_ids          750
duplicate_job_ids         0
missing_job_ids           0
missing_titles            0
missing_descriptions      0
entry_level_jobs          9
skill_mentions_total    149
dtype: int64

In [51]:
enriched_df[
    [
        "id",
        "search_role",
        "title",
        "company",
        "state",
        "salary_midpoint",
        "salary_source",
        "entry_level_flag",
        "career_stage_signal",
    ]
    + skill_columns[:6]
].head(10)

,id,search_role,title,company,state,salary_midpoint,salary_source,entry_level_flag,career_stage_signal,skill_sql,skill_python,skill_excel,skill_tableau,skill_power_bi,skill_looker
0,5901208184,data analyst,Data Analyst,Guidehouse,Alabama,103253.55,predicted,0,unspecified,0,0,0,0,0,0
1,5896874781,data analyst,Data analyst,Vaco LLC,Tennessee,120640.00,non_predicted,0,unspecified,0,0,0,0,0,0
2,5886631734,data analyst,Human Resources Data Analyst,Arnold Porter,"Washington, D.C.",96200.00,non_predicted,0,unspecified,0,0,0,0,0,0
3,5886391534,data analyst,Data Analyst (Financials data Analyst),"eSolutionsFirst, LLC",Virginia,65915.28,predicted,0,unspecified,0,0,0,0,0,0
4,5888579132,data analyst,Data Analyst,MANTECH,Georgia,78684.16,predicted,0,unspecified,0,0,0,0,0,0
5,5823867997,data analyst,Data Analyst,MANTECH,Florida,82150.41,predicted,0,unspecified,0,0,0,0,0,0
6,5877658057,data analyst,Data Analyst,Bet365,Colorado,105242.86,predicted,0,unspecified,0,0,0,0,0,0
7,5901431279,data analyst,Security Assistance Data Analyst,Guidehouse,Virginia,125443.02,predicted,0,unspecified,0,0,0,0,0,0
8,5869673291,data analyst,Data Analyst,Changeis Inc,None,95794.01,predicted,0,unspecified,0,0,0,0,0,0
9,5893069862,data analyst,Data Analyst,Kforce Technology Staffing,Utah,118929.93,predicted,0,unspecified,0,0,0,0,0,0


## 12. Export Enriched Analytical Tables

Export both the enriched job table and the long-form skill table.  
The `data/processed/` directory should remain excluded from Git.

In [52]:
processed_dir.mkdir(parents=True, exist_ok=True)
snapshot_date = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")

enriched_parquet = processed_dir / f"us_analytics_jobs_enriched_{snapshot_date}.parquet"
enriched_csv = processed_dir / f"us_analytics_jobs_enriched_{snapshot_date}.csv"
skills_parquet = processed_dir / f"us_analytics_job_skills_{snapshot_date}.parquet"
skills_csv = processed_dir / f"us_analytics_job_skills_{snapshot_date}.csv"

enriched_df.to_parquet(enriched_parquet, index=False, engine="pyarrow")
enriched_df.to_csv(enriched_csv, index=False)

skill_long_df.to_parquet(skills_parquet, index=False, engine="pyarrow")
skill_long_df.to_csv(skills_csv, index=False)

print("Enriched Parquet:", enriched_parquet)
print("Enriched CSV:", enriched_csv)
print("Skill Parquet:", skills_parquet)
print("Skill CSV:", skills_csv)

Enriched Parquet: ../data/processed/us_analytics_jobs_enriched_2026-09-29.parquet
Enriched CSV: ../data/processed/us_analytics_jobs_enriched_2026-09-29.csv
Skill Parquet: ../data/processed/us_analytics_job_skills_2026-09-29.parquet
Skill CSV: ../data/processed/us_analytics_job_skills_2026-09-29.csv


## 13. Summary

This notebook adds an auditable text-enrichment layer to the U.S. Analytics Job Market Intelligence project.

### Skill extraction
Flags are created for SQL, Python, Excel, Tableau, Power BI, Looker, AWS, GCP, Azure, Snowflake, Spark, PySpark, Databricks, Git, and R.

### Entry-level classification
The heuristic combines explicit new-grad / entry-level language, low experience requirements, higher experience signals, senior title language, and internship language. Component flags and the classification basis are retained for QA.

### Outputs
- **Job-level enriched table** — one row per job with skills, salary, location, and career-stage signals
- **Long-form skill table** — one row per job-skill match for easier BigQuery analysis

### Next step
Load the enriched outputs into BigQuery and use SQL to analyze job demand, skills, salary differences, location patterns, and entry-level opportunities.

Data source: Adzuna API.